# Answer Questions about the 1914 Press

An assistant that answers questions from the 64,000 passages of the 1914 press, citing its sources (a RAG), and its evaluation on 20 questions whose answer, and the passage that gives it, are known. The corpus and the questions are in French, and so are the assistant's answers: a 4-billion-parameter model sticks to the language of its sources, whatever its instructions say.

1. Retrieval: hybrid, then reranking (the building blocks of the product search lab)
2. Evaluate retrieval
3. Generate a sourced answer, or decline
4. Change a setting, and measure
5. Bonus: an LLM judges the answers' faithfulness

This notebook needs a GPU: *Runtime > Change runtime type > T4 GPU*.

## Data and Models

The passages, their precomputed *embeddings* (512 dimensions), the questions; Ollama installs and downloads the model in the background.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/questions.jsonl
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q bm25s

import os
import subprocess

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "8", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pull = subprocess.Popen(f"sleep 5 && ollama pull {MODEL}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

## 1. Retrieval

The search functions are given: they're those of the product search lab. `search(question, k)` returns the indices of the `k` best passages: BM25 and dense retrieval (100 candidates each), fused with RRF, then the first 30 reranked by the cross-encoder (`rerank=False` to skip it). Run the cell (one to two minutes: it also waits for the LLM, and loads it onto the GPU before the other models), and try a question.

In [ ]:
# bm25s uses JAX when it's installed (as on Colab), and JAX grabs 75% of the GPU's memory
# up front: keep it on the CPU, the GPU is for the models.
os.environ["JAX_PLATFORMS"] = "cpu"

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import bm25s
import numpy as np
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel
from sentence_transformers import CrossEncoder, SentenceTransformer

pull.wait()  # the model is downloaded
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# The LLM first: a first call loads it onto the GPU, where it stays, before the other models.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Hello"}], max_tokens=1)
print(MODEL, "ready")

passages = pd.read_parquet("passages.parquet")
vectors = np.load("embeddings-512.npy").astype(np.float32)
questions = pd.read_json("questions.jsonl", lines=True)

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda", truncate_dim=512).half()
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=512)
reranker.model.half()
retriever = bm25s.BM25()
retriever.index(bm25s.tokenize(passages.text.tolist(), show_progress=False), show_progress=False)


def rrf(*rankings, k: int = 60) -> list[int]:
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] += 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)


def search(question: str, k: int = 5, rerank: bool = True) -> list[int]:
    lexical = retriever.retrieve(bm25s.tokenize([question], show_progress=False), k=100, show_progress=False)[0][0]
    query = embedder.encode_query(question).astype(np.float32)
    query /= np.linalg.norm(query)
    dense = np.argsort(-(vectors @ query))[:100]
    candidates = rrf(lexical.tolist(), dense.tolist())
    if not rerank:
        return candidates[:k]
    top = candidates[:30]
    scores = reranker.predict([(question, passages.text[i]) for i in top], batch_size=32)
    return [top[i] for i in np.argsort(-scores)][:k]


for i in search("Où Jaurès a-t-il été assassiné ?", k=3):
    print(passages.id[i], passages.text[i][:200])

## 2. Evaluate Retrieval

For each of the 18 questions with an answer in the corpus, `questions.passage_ids` gives the passage that contains it. Write `hit_rate(k, rerank)`: the share of questions with an expected passage among the first `k` results. Compare with and without reranking, at `k=5`.

In [ ]:
# Your code here

### Solution

In [ ]:
answerable = questions[questions.passage_ids.str.len() > 0]


def hit_rate(k: int = 5, rerank: bool = True) -> float:
    hits = [
        bool(set(passages.id[search(row.question, k, rerank)]) & set(row.passage_ids))
        for row in answerable.itertuples()
    ]
    return float(np.mean(hits))


for rerank in (False, True):
    print(f"rerank={rerank}: {hit_rate(5, rerank):.0%} of the expected passages in the first 5")

## 3. Generate a Sourced Answer, or Decline

Write `answer(question)` that:

- retrieves the 5 best passages, numbered 1 to 5 in the prompt, with their date and newspaper
- asks the model for an answer in the `Answer` format (below): the answer, the numbers of the passages that support it, and `found=False` if the passages don't contain the answer
- instruction: answer **only from the passages**, in French, in one or two sentences

Run it on the 20 questions in parallel, and compare its answers with the reference answers (in French). Are the two unanswerable questions declined?

In [ ]:
class Answer(BaseModel):
    found: bool
    answer: str
    citations: list[int]

In [ ]:
# Your code here

### Solution

In [ ]:
INSTRUCTIONS = """You answer questions about the 1914 French press, only from the numbered
passages provided. Answer in French, like the passages, in one or two sentences, and put the
numbers of the passages that support your answer in citations (not in the answer itself). If
the passages don't contain the answer, set found to false and don't invent anything."""


def answer(question: str, k: int = 5) -> dict:
    found = search(question, k)
    context = "\n\n".join(
        f"[{n}] ({passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y}) {passages.text[i]}"
        for n, i in enumerate(found, start=1)
    )
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": f"Passages:\n{context}\n\nQuestion: {question}"},
        ],
        response_format=Answer,
        temperature=0,
    )
    result = response.choices[0].message.parsed.model_dump()
    result["sources"] = [passages.id[found[n - 1]] for n in result["citations"] if 1 <= n <= len(found)]
    return result


with ThreadPoolExecutor(8) as pool:
    answers = list(pool.map(answer, questions.question))
for row, result in zip(questions.itertuples(), answers):
    print(f"Q{row.id} {row.question}\n  reference: {row.answer}\n  answer:    {result['answer'] if result['found'] else '(declined)'}\n")

## 4. Change a Setting, and Measure

A RAG is tuned like a model: by measuring. Measure the hit rate for `k` = 1, 3, 5 and 10. Which `k` would you choose, knowing that each added passage makes the prompt longer (cost, latency)?

In [ ]:
# Your code here

### Solution

In [ ]:
for k in (1, 3, 5, 10):
    print(f"k={k:2}: {hit_rate(k):.0%}")

## 5. Bonus: an LLM Judges Faithfulness

For each answer, ask the model to judge, from the cited passages and the reference answer: is the answer **faithful** to the passages (nothing invented)? Is it **correct**? Count. Why be wary of a model judging its own answers?

In [ ]:
# Your code here

### Solution

In [ ]:
class Verdict(BaseModel):
    faithful: bool
    correct: bool


def judge(question: str, reference: str | None, result: dict) -> dict:
    cited = "\n\n".join(passages.set_index("id").text[source] for source in result["sources"])
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": (
            f"Question: {question}\nReference answer: {reference or 'none, the question has no answer'}\n"
            f"Cited passages:\n{cited or '(none)'}\n\nAnswer to judge: "
            f"{result['answer'] if result['found'] else '(declined)'}\n\n"
            "faithful: the answer says nothing the cited passages don't say. "
            "correct: the answer says the same as the reference (declining is correct if the question has no answer).\n"
            # The format spelled out: the model doesn't see the schema, and judges worse without it.
            'Answer with a JSON object: {"faithful": true or false, "correct": true or false}'
        )}],
        response_format=Verdict,
        temperature=0,
    )
    return response.choices[0].message.parsed.model_dump()


with ThreadPoolExecutor(8) as pool:
    verdicts = list(pool.map(judge, questions.question, questions.answer, answers))
print(pd.DataFrame(verdicts).mean())

# The same model generates and judges: it tends to approve its own answers (self-preference
# bias). In practice, judge with another, stronger model, and check a sample of the
# verdicts by hand.